In [1]:
import sys
import os
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from src.ingestion.chunker import chunk_markdown_documents
from src.ingestion.indexer import index_documents
from src.agents.graph import create_agent_graph
from langchain_core.documents import Document

collection_name = "apple_10k_reports" # İsim böyle kalsın sorun değil

# 1. VERİTABANINA YENİ VERİLER (METADATA İLE) EKLİYORUZ
print("--- VERİ EKLENİYOR ---")
# Kendi Apple PDF'ini yeniden oku ve etiketle (pdf_parser'dan dönen documents listesini verdiğini varsayalım,
# hızlı test için manuel Document objeleri oluşturuyoruz)

doc_apple = Document(page_content="The common stock, $0.00001 par value per share for Apple.")
doc_tesla = Document(page_content="The common stock par value for Tesla is $0.001 per share.")

# Etiketli Chunk'lar oluşturuyoruz
chunks_apple = chunk_markdown_documents([doc_apple], metadata={"ticker": "AAPL", "year": "2022"})
chunks_tesla = chunk_markdown_documents([doc_tesla], metadata={"ticker": "TSLA", "year": "2022"})

# Qdrant'a yazıyoruz
index_documents(chunks_apple + chunks_tesla, collection_name)


# 2. AJANLARI ÇALIŞTIRIP FİLTREYİ TEST EDİYORUZ
print("\n--- AJANLAR TEST EDİLİYOR ---")
app = create_agent_graph()

# Soru özellikle Tesla'yı soruyor
soru = "What is the common stock par value for Tesla?"
print(f"Soru: {soru}")

sonuc = app.invoke({"question": soru})
print("\n--- NİHAİ CEVAP ---")
print(sonuc["final_answer"])

--- VERİ EKLENİYOR ---
✅ Metin başarıyla 1 parçaya bölündü ve {'ticker': 'AAPL', 'year': '2022'} etiketleri eklendi.
✅ Metin başarıyla 1 parçaya bölündü ve {'ticker': 'TSLA', 'year': '2022'} etiketleri eklendi.
Veriler vektörleştirilip Qdrant'a kaydediliyor...
✅ Veriler başarıyla indekslendi!

--- AJANLAR TEST EDİLİYOR ---
Soru: What is the common stock par value for Tesla?
🧠 Planner: Soru analiz ediliyor, plan ve hedef şirket çıkarılıyor...
🎯 Hedef Şirket Tespit Edildi: TSLA
📚 Retriever: Qdrant veritabanında aranıyor (Filtre: TSLA)...
🔍 Sadece 'TSLA' etiketli belgelerde aranıyor...
🌐 Web Researcher: Hangi aracı kullanması gerektiğine karar veriyor ve araştırıyor...
✍️ Synthesizer: Veriler birleştiriliyor (Deneme: 1)...
🕵️‍♂️ Validator: Rakamlar GERÇEK kaynaklarda aranıyor...
✅ Validator: Tüm sayılar doğru, onaylandı!

--- NİHAİ CEVAP ---
Tesla'nın ortak hisse senedi nominal değeri (par value) hisse başına 0.001 USD'dir.
